In [6]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Sequence, Tuple
import numpy as np

In [7]:
@dataclass
class CTCResult:
    neg_log_likelihood: float          # O_ML({(x,l)}, N_w) = -ln p(l|x)
    dO_du: np.ndarray                  # Eq. 16
    alpha_hat: np.ndarray
    beta_hat: np.ndarray
    C: np.ndarray
    D: np.ndarray
    lprime: List[int]

In [8]:
def make_lprime(l: Sequence[int], blank: int) -> List[int]:
    # |l'| = 2|l| + 1
    # blanks added to the beginning and the end and
    # inserted between every pair of labels.
    lp = [blank]
    for sym in l:
        lp.append(sym)
        lp.append(blank)
    return lp


In [9]:
def ctc_forward_backward(y: np.ndarray, l: Sequence[int], blank: int = 0) -> CTCResult:

  # Preprocess
  # check_labels_softmax_prob(y)
  y = np.asarray(y, dtype=np.float64)
  if y.ndim != 2:
      raise ValueError("y must be 2D (T,K) softmax probabilities")
  if np.any(y < 0) or np.any(y > 1):
        raise ValueError("y should be probabilities in [0,1]")
  if not np.allclose(np.sum(y, axis=1), 1.0, atol=1e-6):
        raise ValueError("each row of y must sum to 1 (softmax probabilities)")

  T, K = y.shape
  if T == 0:
    raise ValueError("T must be > 0")

  lprime = make_lprime(l, blank)
  S = len(lprime)  # corresponds to |l'|

  # Feedforward
  alpha_hat = np.zeros((T, S), dtype=np.float64)
  alpha = np.zeros((S,), dtype=np.float64)
  C = np.zeros((T,), dtype=np.float64)

  alpha[0] = y[0, blank]

  if S > 1:
    alpha[1] = y[0, lprime[1]]

  C[0] = np.sum(alpha)
  if C[0] == 0.0:
    raise FloatingPointError("C[0] is zero; y has zero probability mass on valid starts")

  alpha_hat[0, :] = alpha / C[0]

  for t in range(1, T):
    prev = alpha_hat[t - 1, :]
    curr = np.zeros((S,), dtype=np.float64)

    for s in range(S):
    # alpha_hat_t(s) def = alpha_{t-1}(s) + alpha_{t-1}(s-1)  (Eq. 7)
      a_bar = prev[s]
      if s - 1 >= 0:
        a_bar += prev[s - 1]

      sym_s = lprime[s]
      if sym_s == blank or (s - 2 >= 0 and lprime[s - 2] == sym_s):
        total = a_bar
      else:
        # allow skip from s-2
        total = a_bar + (prev[s - 2] if s - 2 >= 0 else 0.0)

      curr[s] = total * y[t, sym_s]

    C[t] = np.sum(curr)
    if C[t] == 0.0:
      raise FloatingPointError(f"C[{t}] is zero; underflow or impossible alignment")

    alpha_hat[t, :] = curr / C[t]

    # ln p(l|x) = Σ_t ln(C_t)  (paper, after rescaling)
  ln_p = np.sum(np.log(C))
  neg_log_likelihood = -ln_p  # O_ML({(x,l)}, N_w)

  # Backward pass
  beta_hat  = np.zeros((T, S), dtype=np.float64)
  beta = np.zeros((S,), dtype=np.float64)
  D = np.zeros((T,), dtype=np.float64)

  beta[S - 1] = y[T - 1, blank]
  if S - 2 >= 0:
    beta[S - 2] = y[T - 1, lprime[S - 2]]
  D[T - 1] = np.sum(beta)
  if D[T - 1] == 0.0:
    raise FloatingPointError("D[T-1] is zero; y has zero probability mass on valid ends")
  beta_hat[T - 1, :] = beta / D[T - 1]

  for t in range(T - 2, -1, -1):
    nxt = beta_hat[t + 1, :]
    curr = np.zeros((S,), dtype=np.float64)

    for s in range(S):
      # beta_hat_t(s) def = beta_{t+1}(s) + beta_{t+1}(s+1)
      b_bar = nxt[s]
      if s + 1 < S:
        b_bar += nxt[s + 1]

      sym_s = lprime[s]
      if sym_s == blank or (s + 2 < S and lprime[s + 2] == sym_s):
        total = b_bar
      else:
        total = b_bar + (nxt[s + 2] if s + 2 < S else 0.0)

      curr[s] = total * y[t, sym_s]

    D[t] = np.sum(curr)
    if D[t] == 0.0:
      raise FloatingPointError(f"D[{t}] is zero; underflow or impossible alignment")
    beta_hat[t, :] = curr / D[t]

  ratio = D / C  # elementwise D_t / C_t
  suffix = np.ones((T + 1,), dtype=np.float64)
  for t in range(T - 1, -1, -1):
    suffix[t] = suffix[t + 1] * ratio[t]

  Q = np.zeros((T,), dtype=np.float64)
  for t in range(T):
    Q[t] = D[t] * suffix[t + 1]


  dO_du = np.zeros((T, K), dtype=np.float64)

  # Precompute positions for each k
  pos_by_k: List[List[int]] = [[] for _ in range(K)]
  for s, sym in enumerate(lprime):
    pos_by_k[sym].append(s)

  for t in range(T):
    # base term: y_t^k
    dO_du[t, :] = y[t, :]

    for k in range(K):
      if not pos_by_k[k]:
        continue
      summation = 0.0
      for s in pos_by_k[k]:
        summation += alpha_hat[t, s] * beta_hat[t, s]
      dO_du[t, k] -= Q[t] * y[t, k] * summation

  return CTCResult(
        neg_log_likelihood=neg_log_likelihood,
        dO_du=dO_du,
        alpha_hat=alpha_hat,
        beta_hat=beta_hat,
        C=C,
        D=D,
        lprime=lprime,
    )



In [11]:
import numpy as np

blank = 0
A = 1
B = 2

# y_t^k  (T=3, K=3)
y = np.array([
    [0.6, 0.3, 0.1],   # t = 1
    [0.2, 0.5, 0.3],   # t = 2
    [0.1, 0.3, 0.6],   # t = 3
], dtype=np.float64)

l = [A, B]

res = ctc_forward_backward(y, l, blank=blank)

print("Modified label sequence l′:", res.lprime)
print("Negative log-likelihood O_ML:", res.neg_log_likelihood)

print("\nC_t (forward scaling constants):")
print(res.C)

print("\nD_t (backward scaling constants):")
print(res.D)

print("\nalpha_hat_t(s) (forward variables):")
print(res.alpha_hat)

print("\nbeta_hat_t(s) (backward variables):")
print(res.beta_hat)

print("\ngradient, Eq. 16:")
print(res.dO_du)

Modified label sequence l′: [0, 1, 0, 2, 0]
Negative log-likelihood O_ML: 0.5058380822549517

C_t (forward scaling constants):
[0.9    0.8    0.8375]

D_t (backward scaling constants):
[0.92615385 0.92857143 0.7       ]

alpha_hat_t(s) (forward variables):
[[0.66666667 0.33333333 0.         0.         0.        ]
 [0.16666667 0.625      0.08333333 0.125      0.        ]
 [0.0199005  0.28358209 0.08457711 0.59701493 0.01492537]]

beta_hat_t(s) (backward variables):
[[0.29900332 0.31395349 0.32890365 0.03820598 0.01993355]
 [0.         0.46153846 0.18461538 0.32307692 0.03076923]
 [0.         0.         0.         0.85714286 0.14285714]]

gradient, Eq. 16:
[[0.49253731 0.27179104 0.1       ]
 [0.19761194 0.3880597  0.29059701]
 [0.09985075 0.3        0.38507463]]
